این فایل کد حاوی پیاده سازی هم باdistilgpt2 و gpt2

# Decoder with English dataset

# **distilgpt2 Model**

In [1]:
!pip install transformers datasets torch seqeval scikit-learn


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 12.3 MB/s eta 0:00:00
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16161 sha256=d9ffa231584cde92f378bba7199e0fcb9a01ec939ca617d8f57734e5fa9bf8c8
  Stored in directory: /root/.cache/pip/wheels/1a/67/4a/ad4082dd7dfc30f2abfe4d80a2ed5926a506eb8a972b4767fa
Successfully built seqeval
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency 

In [2]:
import json
import torch
import numpy as np
from transformers import GPT2TokenizerFast, GPT2LMHeadModel, Trainer, TrainingArguments
from datasets import Dataset, DatasetDict
from seqeval.metrics import f1_score as seq_f1, precision_score as seq_precision, recall_score as seq_recall, classification_report
from sklearn.metrics import f1_score as sklearn_f1, precision_score as sklearn_precision, recall_score as sklearn_recall
import random
from sklearn.model_selection import train_test_split
from collections import Counter

In [4]:
tokenizer = GPT2TokenizerFast.from_pretrained('distilgpt2')
tokenizer.add_special_tokens({'pad_token': '[PAD]'})
model = GPT2LMHeadModel.from_pretrained('distilgpt2')
model.resize_token_embeddings(len(tokenizer))
model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50258, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-5): 6 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2SdpaAttention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50258, bias=False)
)

## BIOایجاد توکن‌ها و برچسب‌ها با استفاده از برچسب گذاری

In [10]:
def create_tokens_and_labels(id, sample):
    intent = sample['intent']
    utt = sample['utt']
    annot_utt = sample['annot_utt']
    tokens = utt.split()
    labels = []
    label = 'O'
    split_annot_utt = annot_utt.split()
    idx = 0
    BIO_SLOT = False
    while idx < len(split_annot_utt):
        if split_annot_utt[idx].startswith('['):
            label = split_annot_utt[idx].lstrip('[')
            idx += 2
            BIO_SLOT = True
        elif split_annot_utt[idx].endswith(']'):
            if split_annot_utt[idx-1] == ":":
                labels.append("B-" + label)
                label = 'O'
                idx += 1
            else:
                labels.append("I-" + label)
                label = 'O'
                idx += 1
            BIO_SLOT = False
        else:
            if split_annot_utt[idx-1] == ":":
                labels.append("B-" + label)
                idx += 1
            elif BIO_SLOT:
                labels.append("I-" + label)
                idx += 1
            else:
                labels.append("O")
                idx += 1

    if len(tokens) != len(labels):
        raise ValueError(f"Length of tokens, {tokens}, doesn't match length of labels, {labels}, "
                         f"for id {id} and annot_utt: {annot_utt}")
    return tokens, labels, intent

sentences_tr, tags_tr, intent_tags_tr = [], [], []
sentences_val, tags_val, intent_tags_val = [], [], []
sentences_test, tags_test, intent_tags_test = [], [], []

massive_raw = []
with open('/content/drive/MyDrive/en-US.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        massive_raw.append(json.loads(line))

for id, sample in enumerate(massive_raw):
    tokens, labels, intent = create_tokens_and_labels(id, sample)
    if sample['partition'] == 'train':
        sentences_tr.append(tokens)
        tags_tr.append(labels)
        intent_tags_tr.append(intent)
    elif sample['partition'] == 'dev':
        sentences_val.append(tokens)
        tags_val.append(labels)
        intent_tags_val.append(intent)
    elif sample['partition'] == 'test':
        sentences_test.append(tokens)
        tags_test.append(labels)
        intent_tags_test.append(intent)


sentences_tr, _, tags_tr, _, intent_tags_tr, _ = train_test_split(
    sentences_tr, tags_tr, intent_tags_tr, test_size=0.9, random_state=42)

val_indices = random.sample(range(len(sentences_val)), 200)
sentences_val = [sentences_val[i] for i in val_indices]
tags_val = [tags_val[i] for i in val_indices]
intent_tags_val = [intent_tags_val[i] for i in val_indices]


test_indices = random.sample(range(len(sentences_test)), 200)
sentences_test = [sentences_test[i] for i in test_indices]
tags_test = [tags_test[i] for i in test_indices]
intent_tags_test = [intent_tags_test[i] for i in test_indices]

partition_counts = {'train': len(sentences_tr), 'test': len(sentences_test), 'dev': len(sentences_val)}
print("تعداد نمونه‌ها در هر بخش:", partition_counts)


تعداد نمونه‌ها در هر بخش: {'train': 1151, 'test': 200, 'dev': 200}


**استخراج لیست برچسب‌ها قصد و شکاف**

In [11]:
unique_slot_labels = set(label for sublist in tags_tr + tags_val + tags_test for label in sublist)
unique_intent_labels = set(intent_tags_tr + intent_tags_val + intent_tags_test)

label_list = sorted(unique_slot_labels.union(unique_intent_labels))

print("\nلیست برچسب‌های استخراج شده به صورت خودکار:")
print(label_list)


لیست برچسب‌های استخراج شده به صورت خودکار:
['B-alarm_type', 'B-app_name', 'B-artist_name', 'B-audiobook_author', 'B-audiobook_name', 'B-business_name', 'B-business_type', 'B-change_amount', 'B-color_type', 'B-cooking_type', 'B-currency_name', 'B-date', 'B-definition_word', 'B-device_type', 'B-drink_type', 'B-email_address', 'B-email_folder', 'B-event_name', 'B-food_type', 'B-game_name', 'B-general_frequency', 'B-house_place', 'B-ingredient', 'B-joke_type', 'B-list_name', 'B-meal_type', 'B-media_type', 'B-movie_name', 'B-movie_type', 'B-music_descriptor', 'B-music_genre', 'B-news_topic', 'B-order_type', 'B-person', 'B-personal_info', 'B-place_name', 'B-player_setting', 'B-playlist_name', 'B-podcast_descriptor', 'B-podcast_name', 'B-radio_name', 'B-relation', 'B-song_name', 'B-time', 'B-time_zone', 'B-timeofday', 'B-transport_agency', 'B-transport_descriptor', 'B-transport_type', 'B-weather_descriptor', 'I-alarm_type', 'I-artist_name', 'I-audiobook_author', 'I-audiobook_name', 'I-busine

**تابع برای ایجاد پرامپ**

In [12]:
def create_prompt(sentence, intent, slots):

    sentence_text = ' '.join(sentence)
    slots_text = ' '.join(slots)
    prompt = f'Input: {sentence_text}\nOutput Intent: {intent}\nOutput Slots: {slots_text}'
    return prompt

train_prompts = [create_prompt(sent, intent, slots) for sent, intent, slots in zip(sentences_tr, intent_tags_tr, tags_tr)]

val_prompts = [create_prompt(sent, intent, slots) for sent, intent, slots in zip(sentences_val, intent_tags_val, tags_val)]

test_prompts = [create_prompt(sent, intent, slots) for sent, intent, slots in zip(sentences_test, intent_tags_test, tags_test)]

print("نمونه‌های پرامپت‌های آموزشی:")
for i in range(2):
    print(f"Training Prompt {i+1}:")
    print(train_prompts[i])
    print("\n")

print("نمونه‌های پرامپت‌های اعتبارسنجی:")
for i in range(2):
    print(f"Validation Prompt {i+1}:")
    print(val_prompts[i])
    print("\n")

print("نمونه‌های پرامپت‌های تست:")
for i in range(2):
    print(f"Test Prompt {i+1}:")
    print(test_prompts[i])
    print("\n")


نمونه‌های پرامپت‌های آموزشی:
Training Prompt 1:
Input: please remove dinner date scheduled for this friday at nine p. m.
Output Intent: calendar_remove
Output Slots: O O B-meal_type O O O O B-date O B-time I-time I-time


Training Prompt 2:
Input: read freds emails
Output Intent: email_query
Output Slots: O B-person O


نمونه‌های پرامپت‌های اعتبارسنجی:
Validation Prompt 1:
Input: make a new list
Output Intent: lists_createoradd
Output Slots: O O O O


Validation Prompt 2:
Input: power off the current
Output Intent: iot_hue_lightoff
Output Slots: O O O O


نمونه‌های پرامپت‌های تست:
Test Prompt 1:
Input: please be quiet for another hour
Output Intent: audio_volume_mute
Output Slots: O O O O B-time I-time


Test Prompt 2:
Input: where is yosemite park
Output Intent: qa_factoid
Output Slots: O O B-place_name I-place_name




# distilgpt2پیش پردازش وتوکنایز داده‌ها با مدل پیش‌آموزش دیده

In [13]:
def tokenize_function(examples):

    tokenized = tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=70,
        return_offsets_mapping=True
    )

    labels = []
    for i, slot_labels in enumerate(examples['slot_labels']):
        label_ids = [-100] * len(tokenized['input_ids'][i])


        output_slots_prefix = 'Output Slots: '
        output_slots_prefix_ids = tokenizer.encode(output_slots_prefix, add_special_tokens=False)

        try:

            prefix_start = tokenized['input_ids'][i].index(output_slots_prefix_ids[0])
            for j in range(len(output_slots_prefix_ids)):
                label_ids[prefix_start + j] = -100
            slot_start = prefix_start + len(output_slots_prefix_ids)


            for k, slot in enumerate(slot_labels):
                if slot in label_list:
                    label_id = label_list.index(slot)
                    if slot_start + k < len(label_ids):
                        label_ids[slot_start + k] = label_id
                else:
                    label_ids[slot_start + k] = -100
        except ValueError:

            pass

        labels.append(label_ids)

    tokenized['labels'] = labels
    return tokenized

# تبدیل پرامپت‌ها به دیکشنری همراه با برچسب‌ها
train_data = {'text': train_prompts, 'slot_labels': tags_tr}
val_data = {'text': val_prompts, 'slot_labels': tags_val}
test_data = {'text': test_prompts, 'slot_labels': tags_test}

train_dataset = Dataset.from_dict(train_data)
val_dataset = Dataset.from_dict(val_data)
test_dataset = Dataset.from_dict(test_data)


dataset = DatasetDict({
    'train': train_dataset,
    'validation': val_dataset,
    'test': test_dataset
})

tokenized_datasets = dataset.map(tokenize_function, batched=True)



Map:   0%|          | 0/1151 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

# تعریف تابع برای محاسبه متریک‌ها در آموزش

In [14]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=-1)


    preds = preds.tolist()
    labels = labels.tolist()

    true_labels = []
    true_preds = []

    for pred, label in zip(preds, labels):
        temp_true_labels = []
        temp_true_preds = []
        for p, l in zip(pred, label):
            if l != -100:
                temp_true_labels.append(l)
                temp_true_preds.append(p)
        true_labels.append(temp_true_labels)
        true_preds.append(temp_true_preds)

    true_labels_text = [[label_list[l] for l in labels] for labels in true_labels]
    preds_text = [[label_list[p] for p in preds] for preds in true_preds]

    slot_f1_micro = seq_f1(true_labels_text, preds_text, average='micro')
    slot_precision_micro = seq_precision(true_labels_text, preds_text, average='micro')
    slot_recall_micro = seq_recall(true_labels_text, preds_text, average='micro')


    true_intents = [labels[-1] for labels in true_labels if len(labels) > 0]
    pred_intents = [preds[-1] for preds in true_preds if len(preds) > 0]

    true_intents_text = [label_list[l] for l in true_intents]
    pred_intents_text = [label_list[p] for p in pred_intents]

    intent_f1_micro = sklearn_f1(true_intents_text, pred_intents_text, average='micro')
    intent_precision_micro = sklearn_precision(true_intents_text, pred_intents_text, average='micro')
    intent_recall_micro = sklearn_recall(true_intents_text, pred_intents_text, average='micro')

    return {
        'slot_f1_micro': slot_f1_micro,
        'slot_precision_micro': slot_precision_micro,
        'slot_recall_micro': slot_recall_micro,
        'intent_f1_micro': intent_f1_micro,
        'intent_precision_micro': intent_precision_micro,
        'intent_recall_micro': intent_recall_micro,
    }

# آموزش

In [15]:
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    logging_dir='./logs',
    logging_steps=50,
    save_total_limit=2,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    fp16=True,
    gradient_checkpointing=True,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to=[],
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)


trainer.train()
torch.cuda.empty_cache()

trainer.save_model('./trained_gpt2_model')
torch.cuda.empty_cache()


/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
<ipython-input-15-07e5b957a675>:22: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Slot F1 Micro,Slot Precision Micro,Slot Recall Micro,Intent F1 Micro,Intent Precision Micro,Intent Recall Micro
1,4.941100,1.152707,0.022989,0.051724,0.014778,0.515000,0.515000,0.515000
2,3.602900,1.017608,0.087209,0.106383,0.073892,0.515000,0.515000,0.515000
3,3.338300,0.865675,0.125392,0.172414,0.098522,0.545000,0.545000,0.545000
4,2.417400,0.808129,0.150754,0.153846,0.147783,0.565000,0.565000,0.565000


There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


# ارزیابی مدل

In [16]:
def get_labels(predictions, label_ids):

    preds = np.argmax(predictions, axis=-1)
    labels = label_ids

    preds = preds.tolist()
    labels = labels.tolist()

    true_labels = []
    true_preds = []

    for pred, label in zip(preds, labels):
        temp_true_labels = []
        temp_true_preds = []
        for p, l in zip(pred, label):
            if l != -100:
                temp_true_labels.append(l)
                temp_true_preds.append(p)
        true_labels.append(temp_true_labels)
        true_preds.append(temp_true_preds)

    return true_labels, true_preds

def id_to_label(id_list, label_map):

    return [[label_map[id] for id in labels] for labels in id_list]

def predict_in_batches(trainer, dataset, batch_size=16):
    predictions = []
    label_ids = []
    for i in range(0, len(dataset), batch_size):
        batch = dataset.select(range(i, min(i + batch_size, len(dataset))))
        results = trainer.predict(batch)
        predictions.append(results.predictions)
        label_ids.append(results.label_ids)
        torch.cuda.empty_cache()
    predictions = np.concatenate(predictions, axis=0)
    label_ids = np.concatenate(label_ids, axis=0)
    return predictions, label_ids

val_predictions, val_label_ids = predict_in_batches(trainer, tokenized_datasets['validation'], batch_size=16)

val_true_labels, val_preds = get_labels(val_predictions, val_label_ids)

val_true_labels_text = id_to_label(val_true_labels, label_list)
val_preds_text = id_to_label(val_preds, label_list)

val_f1_micro = seq_f1(val_true_labels_text, val_preds_text, average='micro')
val_precision_micro = seq_precision(val_true_labels_text, val_preds_text, average='micro')
val_recall_micro = seq_recall(val_true_labels_text, val_preds_text, average='micro')

print("\nValidation Slot Filling Metrics (Micro):")
print(f"Precision: {val_precision_micro:.4f}")
print(f"Recall: {val_recall_micro:.4f}")
print(f"F1 Score: {val_f1_micro:.4f}")

val_f1_macro = seq_f1(val_true_labels_text, val_preds_text, average='macro')
val_precision_macro = seq_precision(val_true_labels_text, val_preds_text, average='macro')
val_recall_macro = seq_recall(val_true_labels_text, val_preds_text, average='macro')

print("\nValidation Slot Filling Metrics (Macro):")
print(f"Precision: {val_precision_macro:.4f}")
print(f"Recall: {val_recall_macro:.4f}")
print(f"F1 Score: {val_f1_macro:.4f}")

print("\nValidation Slot Filling Classification Report:")
print(classification_report(val_true_labels_text, val_preds_text, digits=4))


val_true_intents = [labels[-1] for labels in val_true_labels if len(labels) > 0]
val_pred_intents = [preds[-1] for preds in val_preds if len(preds) > 0]


val_true_intents_text = [label_list[id] for id in val_true_intents]
val_pred_intents_text = [label_list[id] for id in val_pred_intents]

intent_f1_micro = sklearn_f1(val_true_intents_text, val_pred_intents_text, average='micro')
intent_precision_micro = sklearn_precision(val_true_intents_text, val_pred_intents_text, average='micro')
intent_recall_micro = sklearn_recall(val_true_intents_text, val_pred_intents_text, average='micro')

print("\nValidation Intent Detection Metrics (Micro):")
print(f"Precision: {intent_precision_micro:.4f}")
print(f"Recall: {intent_recall_micro:.4f}")
print(f"F1 Score: {intent_f1_micro:.4f}")

intent_f1_macro = sklearn_f1(val_true_intents_text, val_pred_intents_text, average='macro')
intent_precision_macro = sklearn_precision(val_true_intents_text, val_pred_intents_text, average='macro')
intent_recall_macro = sklearn_recall(val_true_intents_text, val_pred_intents_text, average='macro')

print("\nValidation Intent Detection Metrics (Macro):")
print(f"Precision: {intent_precision_macro:.4f}")
print(f"Recall: {intent_recall_macro:.4f}")
print(f"F1 Score: {intent_f1_macro:.4f}")

print("\nValidation Intent Detection Classification Report (sklearn):")
print(sklearn_classification_report(val_true_intents_text, val_pred_intents_text, digits=4))


torch.cuda.empty_cache()

test_predictions, test_label_ids = predict_in_batches(trainer, tokenized_datasets['test'], batch_size=16)

test_true_labels, test_preds = get_labels(test_predictions, test_label_ids)


test_true_labels_text = id_to_label(test_true_labels, label_list)
test_preds_text = id_to_label(test_preds, label_list)


test_f1_micro = seq_f1(test_true_labels_text, test_preds_text, average='micro')
test_precision_micro = seq_precision(test_true_labels_text, test_preds_text, average='micro')
test_recall_micro = seq_recall(test_true_labels_text, test_preds_text, average='micro')

print("\nTest Slot Filling Metrics (Micro):")
print(f"Precision: {test_precision_micro:.4f}")
print(f"Recall: {test_recall_micro:.4f}")
print(f"F1 Score: {test_f1_micro:.4f}")


test_f1_macro = seq_f1(test_true_labels_text, test_preds_text, average='macro')
test_precision_macro = seq_precision(test_true_labels_text, test_preds_text, average='macro')
test_recall_macro = seq_recall(test_true_labels_text, test_preds_text, average='macro')

print("\nTest Slot Filling Metrics (Macro):")
print(f"Precision: {test_precision_macro:.4f}")
print(f"Recall: {test_recall_macro:.4f}")
print(f"F1 Score: {test_f1_macro:.4f}")


print("\nTest Slot Filling Classification Report:")
print(classification_report(test_true_labels_text, test_preds_text, digits=4))


test_true_intents = [labels[-1] for labels in test_true_labels if len(labels) > 0]
test_pred_intents = [preds[-1] for preds in test_preds if len(preds) > 0]

test_true_intents_text = [label_list[id] for id in test_true_intents]
test_pred_intents_text = [label_list[id] for id in test_pred_intents]

test_intent_f1_micro = sklearn_f1(test_true_intents_text, test_pred_intents_text, average='micro')
test_intent_precision_micro = sklearn_precision(test_true_intents_text, test_pred_intents_text, average='micro')
test_intent_recall_micro = sklearn_recall(test_true_intents_text, test_pred_intents_text, average='micro')

print("\nTest Intent Detection Metrics (Micro):")
print(f"Precision: {test_intent_precision_micro:.4f}")
print(f"Recall: {test_intent_recall_micro:.4f}")
print(f"F1 Score: {test_intent_f1_micro:.4f}")

test_intent_f1_macro = sklearn_f1(test_true_intents_text, test_pred_intents_text, average='macro')
test_intent_precision_macro = sklearn_precision(test_true_intents_text, test_pred_intents_text, average='macro')
test_intent_recall_macro = sklearn_recall(test_true_intents_text, test_pred_intents_text, average='macro')

print("\nTest Intent Detection Metrics (Macro):")
print(f"Precision: {test_intent_precision_macro:.4f}")
print(f"Recall: {test_intent_recall_macro:.4f}")
print(f"F1 Score: {test_intent_f1_macro:.4f}")

print("\nTest Intent Detection Classification Report (sklearn):")
print(sklearn_classification_report(test_true_intents_text, test_pred_intents_text, digits=4))

decoded_text = tokenizer.decode(tokenized_datasets['train'][0]['input_ids'], skip_special_tokens=True)
print("\nمتن بازگردانی‌شده:")
print(decoded_text)
original_text = tokenized_datasets['train'][0]['text']
print("\nمتن اصلی:")
print(original_text)



Validation Slot Filling Metrics (Micro):
Precision: 0.1538
Recall: 0.1478
F1 Score: 0.1508

Validation Slot Filling Metrics (Macro):
Precision: 0.1202
Recall: 0.1397
F1 Score: 0.1098

Validation Slot Filling Classification Report:
                    precision    recall  f1-score   support

       artist_name     0.0000    0.0000    0.0000         5
    audiobook_name     0.1429    0.2500    0.1818         4
     business_name     0.1000    0.1429    0.1176         7
     business_type     0.2500    1.0000    0.4000         1
        color_type     1.0000    0.5000    0.6667         2
     currency_name     0.0000    0.0000    0.0000         5
              date     0.1594    0.3333    0.2157        33
   definition_word     0.4286    1.0000    0.6000         3
       device_type     0.0000    0.0000    0.0000         3
        drink_type     0.0000    0.0000    0.0000         1
      email_folder     0.0000    0.0000    0.0000         1
        event_name     0.0000    0.0000    0.00

/usr/local/lib/python3.10/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0


Test Slot Filling Metrics (Micro):
Precision: 0.1596
Recall: 0.1676
F1 Score: 0.1635

Test Slot Filling Metrics (Macro):
Precision: 0.0881
Recall: 0.0922
F1 Score: 0.0791

Test Slot Filling Classification Report:
                    precision    recall  f1-score   support

       artist_name     0.0000    0.0000    0.0000         8
     business_name     0.0000    0.0000    0.0000         4
     business_type     0.0000    0.0000    0.0000         3
      cooking_type     0.0000    0.0000    0.0000         1
     currency_name     0.0000    0.0000    0.0000         1
              date     0.2292    0.4583    0.3056        24
   definition_word     0.4286    0.7500    0.5455         4
       device_type     0.0000    0.0000    0.0000         4
      email_folder     0.0000    0.0000    0.0000         1
        event_name     0.2500    0.2381    0.2439        21
         food_type     0.0000    0.0000    0.0000         5
         game_name     0.0000    0.0000    0.0000         1
     

/usr/local/lib/python3.10/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarnin

# **GPT2 Model**



---



---



---



In [1]:
!pip install transformers datasets torch seqeval scikit-learn


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 17.5 MB/s eta 0:00:00
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16161 sha256=8e5911503c3b5bcd9bdb6d7be9358196bbb41d223c9c5ab2423adecdbbe4dea7
  Stored in directory: /root/.cache/pip/wheels/1a/67/4a/ad4082dd7dfc30f2abfe4d80a2ed5926a506eb8a972b4767fa
Successfully built seqeval
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency 

In [2]:
import json
import torch
import numpy as np
from transformers import GPT2TokenizerFast, GPT2LMHeadModel, Trainer, TrainingArguments
from datasets import Dataset, DatasetDict
from seqeval.metrics import f1_score as seq_f1, precision_score as seq_precision, recall_score as seq_recall, classification_report
from sklearn.metrics import f1_score as sklearn_f1, precision_score as sklearn_precision, recall_score as sklearn_recall
import random
from sklearn.model_selection import train_test_split
from collections import Counter
from transformers import GPT2TokenizerFast, GPT2LMHeadModel, Trainer, TrainingArguments

# مدل GPT2

In [3]:
tokenizer = GPT2TokenizerFast.from_pretrained('gpt2')
tokenizer.add_special_tokens({'pad_token': '[PAD]'})
model = GPT2LMHeadModel.from_pretrained('gpt2')
model.resize_token_embeddings(len(tokenizer))
model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50258, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2SdpaAttention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50258, bias=False)
)

# **BIOایجاد توکن‌ها و برچسب‌ها با استفاده از برچسب گذاری**

In [4]:
def create_tokens_and_labels(id, sample):
    intent = sample['intent']
    utt = sample['utt']
    annot_utt = sample['annot_utt']
    tokens = utt.split()
    labels = []
    label = 'O'
    split_annot_utt = annot_utt.split()
    idx = 0
    BIO_SLOT = False
    while idx < len(split_annot_utt):
        if split_annot_utt[idx].startswith('['):
            label = split_annot_utt[idx].lstrip('[')
            idx += 2
            BIO_SLOT = True
        elif split_annot_utt[idx].endswith(']'):
            if split_annot_utt[idx-1] == ":":
                labels.append("B-" + label)
                label = 'O'
                idx += 1
            else:
                labels.append("I-" + label)
                label = 'O'
                idx += 1
            BIO_SLOT = False
        else:
            if split_annot_utt[idx-1] == ":":
                labels.append("B-" + label)
                idx += 1
            elif BIO_SLOT:
                labels.append("I-" + label)
                idx += 1
            else:
                labels.append("O")
                idx += 1

    if len(tokens) != len(labels):
        raise ValueError(f"Length of tokens, {tokens}, doesn't match length of labels, {labels}, "
                         f"for id {id} and annot_utt: {annot_utt}")
    return tokens, labels, intent

sentences_tr, tags_tr, intent_tags_tr = [], [], []
sentences_val, tags_val, intent_tags_val = [], [], []
sentences_test, tags_test, intent_tags_test = [], [], []

massive_raw = []
with open('/content/drive/MyDrive/en-US.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        massive_raw.append(json.loads(line))

for id, sample in enumerate(massive_raw):
    tokens, labels, intent = create_tokens_and_labels(id, sample)
    if sample['partition'] == 'train':
        sentences_tr.append(tokens)
        tags_tr.append(labels)
        intent_tags_tr.append(intent)
    elif sample['partition'] == 'dev':
        sentences_val.append(tokens)
        tags_val.append(labels)
        intent_tags_val.append(intent)
    elif sample['partition'] == 'test':
        sentences_test.append(tokens)
        tags_test.append(labels)
        intent_tags_test.append(intent)


sentences_tr, _, tags_tr, _, intent_tags_tr, _ = train_test_split(
    sentences_tr, tags_tr, intent_tags_tr, test_size=0.9, random_state=42)

val_indices = random.sample(range(len(sentences_val)), 200)
sentences_val = [sentences_val[i] for i in val_indices]
tags_val = [tags_val[i] for i in val_indices]
intent_tags_val = [intent_tags_val[i] for i in val_indices]

test_indices = random.sample(range(len(sentences_test)), 200)
sentences_test = [sentences_test[i] for i in test_indices]
tags_test = [tags_test[i] for i in test_indices]
intent_tags_test = [intent_tags_test[i] for i in test_indices]

partition_counts = {'train': len(sentences_tr), 'test': len(sentences_test), 'dev': len(sentences_val)}
print("تعداد نمونه‌ها در هر بخش:", partition_counts)




تعداد نمونه‌ها در هر بخش: {'train': 1151, 'test': 200, 'dev': 200}


In [5]:
unique_slot_labels = set(label for sublist in tags_tr + tags_val + tags_test for label in sublist)
unique_intent_labels = set(intent_tags_tr + intent_tags_val + intent_tags_test)

label_list = sorted(unique_slot_labels.union(unique_intent_labels))

print("\nلیست برچسب‌های استخراج شده به صورت خودکار:")
print(label_list)


لیست برچسب‌های استخراج شده به صورت خودکار:
['B-alarm_type', 'B-app_name', 'B-artist_name', 'B-audiobook_author', 'B-audiobook_name', 'B-business_name', 'B-business_type', 'B-change_amount', 'B-color_type', 'B-cooking_type', 'B-currency_name', 'B-date', 'B-definition_word', 'B-device_type', 'B-email_address', 'B-email_folder', 'B-event_name', 'B-food_type', 'B-game_name', 'B-general_frequency', 'B-house_place', 'B-ingredient', 'B-joke_type', 'B-list_name', 'B-meal_type', 'B-media_type', 'B-movie_name', 'B-movie_type', 'B-music_descriptor', 'B-music_genre', 'B-news_topic', 'B-order_type', 'B-person', 'B-personal_info', 'B-place_name', 'B-player_setting', 'B-playlist_name', 'B-podcast_descriptor', 'B-podcast_name', 'B-radio_name', 'B-relation', 'B-song_name', 'B-time', 'B-time_zone', 'B-timeofday', 'B-transport_agency', 'B-transport_descriptor', 'B-transport_name', 'B-transport_type', 'B-weather_descriptor', 'I-alarm_type', 'I-artist_name', 'I-audiobook_author', 'I-audiobook_name', 'I-bu

# create prompt

In [6]:
def create_prompt(sentence, intent, slots):

    sentence_text = ' '.join(sentence)
    slots_text = ' '.join(slots)
    prompt = f'Input: {sentence_text}\nOutput Intent: {intent}\nOutput Slots: {slots_text}'
    return prompt

train_prompts = [create_prompt(sent, intent, slots) for sent, intent, slots in zip(sentences_tr, intent_tags_tr, tags_tr)]

val_prompts = [create_prompt(sent, intent, slots) for sent, intent, slots in zip(sentences_val, intent_tags_val, tags_val)]

test_prompts = [create_prompt(sent, intent, slots) for sent, intent, slots in zip(sentences_test, intent_tags_test, tags_test)]

print("نمونه‌های پرامپت‌های آموزشی:")
for i in range(2):
    print(f"Training Prompt {i+1}:")
    print(train_prompts[i])
    print("\n")


print("نمونه‌های پرامپت‌های اعتبارسنجی:")
for i in range(2):
    print(f"Validation Prompt {i+1}:")
    print(val_prompts[i])
    print("\n")


print("نمونه‌های پرامپت‌های تست:")
for i in range(2):
    print(f"Test Prompt {i+1}:")
    print(test_prompts[i])
    print("\n")


نمونه‌های پرامپت‌های آموزشی:
Training Prompt 1:
Input: please remove dinner date scheduled for this friday at nine p. m.
Output Intent: calendar_remove
Output Slots: O O B-meal_type O O O O B-date O B-time I-time I-time


Training Prompt 2:
Input: read freds emails
Output Intent: email_query
Output Slots: O B-person O


نمونه‌های پرامپت‌های اعتبارسنجی:
Validation Prompt 1:
Input: what is the weather supposed to be like this week
Output Intent: weather_query
Output Slots: O O O O O O O O B-date I-date


Validation Prompt 2:
Input: what will be the temperature today for miami florida
Output Intent: weather_query
Output Slots: O O O O B-weather_descriptor B-date O B-place_name I-place_name


نمونه‌های پرامپت‌های تست:
Test Prompt 1:
Input: what is the timing of bagmati express
Output Intent: transport_query
Output Slots: O O O O O B-transport_name I-transport_name


Test Prompt 2:
Input: my day is going well add a memo
Output Intent: general_quirky
Output Slots: O O O O O O O O




# پیش پردازش و توکنایز

In [7]:
def tokenize_function(examples):

    tokenized = tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=70,
        return_offsets_mapping=True
    )

    labels = []
    for i, slot_labels in enumerate(examples['slot_labels']):
        label_ids = [-100] * len(tokenized['input_ids'][i])

        output_slots_prefix = 'Output Slots: '
        output_slots_prefix_ids = tokenizer.encode(output_slots_prefix, add_special_tokens=False)

        try:

            prefix_start = tokenized['input_ids'][i].index(output_slots_prefix_ids[0])
            for j in range(len(output_slots_prefix_ids)):
                label_ids[prefix_start + j] = -100
            slot_start = prefix_start + len(output_slots_prefix_ids)

            for k, slot in enumerate(slot_labels):
                if slot in label_list:
                    label_id = label_list.index(slot)
                    if slot_start + k < len(label_ids):
                        label_ids[slot_start + k] = label_id
                else:
                    label_ids[slot_start + k] = -100
        except ValueError:

            pass

        labels.append(label_ids)

    tokenized['labels'] = labels
    return tokenized

train_data = {'text': train_prompts, 'slot_labels': tags_tr}
val_data = {'text': val_prompts, 'slot_labels': tags_val}
test_data = {'text': test_prompts, 'slot_labels': tags_test}


train_dataset = Dataset.from_dict(train_data)
val_dataset = Dataset.from_dict(val_data)
test_dataset = Dataset.from_dict(test_data)


dataset = DatasetDict({
    'train': train_dataset,
    'validation': val_dataset,
    'test': test_dataset
})

tokenized_datasets = dataset.map(tokenize_function, batched=True)



Map:   0%|          | 0/1151 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

In [8]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=-1)

    preds = preds.tolist()
    labels = labels.tolist()

    true_labels = []
    true_preds = []

    for pred, label in zip(preds, labels):
        temp_true_labels = []
        temp_true_preds = []
        for p, l in zip(pred, label):
            if l != -100:
                temp_true_labels.append(l)
                temp_true_preds.append(p)
        true_labels.append(temp_true_labels)
        true_preds.append(temp_true_preds)

    true_labels_text = [[label_list[l] for l in labels] for labels in true_labels]
    preds_text = [[label_list[p] for p in preds] for preds in true_preds]

    slot_f1_micro = seq_f1(true_labels_text, preds_text, average='micro')
    slot_precision_micro = seq_precision(true_labels_text, preds_text, average='micro')
    slot_recall_micro = seq_recall(true_labels_text, preds_text, average='micro')


    true_intents = [labels[-1] for labels in true_labels if len(labels) > 0]
    pred_intents = [preds[-1] for preds in true_preds if len(preds) > 0]


    true_intents_text = [label_list[l] for l in true_intents]
    pred_intents_text = [label_list[p] for p in pred_intents]


    intent_f1_micro = sklearn_f1(true_intents_text, pred_intents_text, average='micro')
    intent_precision_micro = sklearn_precision(true_intents_text, pred_intents_text, average='micro')
    intent_recall_micro = sklearn_recall(true_intents_text, pred_intents_text, average='micro')

    return {
        'slot_f1_micro': slot_f1_micro,
        'slot_precision_micro': slot_precision_micro,
        'slot_recall_micro': slot_recall_micro,
        'intent_f1_micro': intent_f1_micro,
        'intent_precision_micro': intent_precision_micro,
        'intent_recall_micro': intent_recall_micro,
    }

# آموزش

In [9]:
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    logging_dir='./logs',
    logging_steps=50,
    save_total_limit=2,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    fp16=True,
    gradient_checkpointing=True,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to=[],
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)


trainer.train()
torch.cuda.empty_cache()

trainer.save_model('./trained_gpt2_model1')
torch.cuda.empty_cache()


/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
<ipython-input-9-d6312d607bda>:22: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Slot F1 Micro,Slot Precision Micro,Slot Recall Micro,Intent F1 Micro,Intent Precision Micro,Intent Recall Micro
1,4.921500,1.221545,0.072034,0.059233,0.091892,0.470000,0.470000,0.470000
2,3.394800,0.938802,0.112994,0.118343,0.108108,0.570000,0.570000,0.570000
3,3.165600,0.828760,0.137931,0.164179,0.118919,0.605000,0.605000,0.605000
4,1.906900,0.822357,0.193906,0.198864,0.189189,0.610000,0.610000,0.610000


There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


# ارزیابی

In [10]:
def get_labels(predictions, label_ids):

    preds = np.argmax(predictions, axis=-1)
    labels = label_ids

    preds = preds.tolist()
    labels = labels.tolist()

    true_labels = []
    true_preds = []

    for pred, label in zip(preds, labels):
        temp_true_labels = []
        temp_true_preds = []
        for p, l in zip(pred, label):
            if l != -100:
                temp_true_labels.append(l)
                temp_true_preds.append(p)
        true_labels.append(temp_true_labels)
        true_preds.append(temp_true_preds)

    return true_labels, true_preds

def id_to_label(id_list, label_map):

    return [[label_map[id] for id in labels] for labels in id_list]

def predict_in_batches(trainer, dataset, batch_size=16):
    predictions = []
    label_ids = []
    for i in range(0, len(dataset), batch_size):
        batch = dataset.select(range(i, min(i + batch_size, len(dataset))))
        results = trainer.predict(batch)
        predictions.append(results.predictions)
        label_ids.append(results.label_ids)
        torch.cuda.empty_cache()
    predictions = np.concatenate(predictions, axis=0)
    label_ids = np.concatenate(label_ids, axis=0)
    return predictions, label_ids

val_predictions, val_label_ids = predict_in_batches(trainer, tokenized_datasets['validation'], batch_size=16)

val_true_labels, val_preds = get_labels(val_predictions, val_label_ids)

val_true_labels_text = id_to_label(val_true_labels, label_list)
val_preds_text = id_to_label(val_preds, label_list)

val_f1_micro = seq_f1(val_true_labels_text, val_preds_text, average='micro')
val_precision_micro = seq_precision(val_true_labels_text, val_preds_text, average='micro')
val_recall_micro = seq_recall(val_true_labels_text, val_preds_text, average='micro')

print("\nValidation Slot Filling Metrics (Micro):")
print(f"Precision: {val_precision_micro:.4f}")
print(f"Recall: {val_recall_micro:.4f}")
print(f"F1 Score: {val_f1_micro:.4f}")

val_f1_macro = seq_f1(val_true_labels_text, val_preds_text, average='macro')
val_precision_macro = seq_precision(val_true_labels_text, val_preds_text, average='macro')
val_recall_macro = seq_recall(val_true_labels_text, val_preds_text, average='macro')

print("\nValidation Slot Filling Metrics (Macro):")
print(f"Precision: {val_precision_macro:.4f}")
print(f"Recall: {val_recall_macro:.4f}")
print(f"F1 Score: {val_f1_macro:.4f}")

print("\nValidation Slot Filling Classification Report:")
print(classification_report(val_true_labels_text, val_preds_text, digits=4))

val_true_intents = [labels[-1] for labels in val_true_labels if len(labels) > 0]
val_pred_intents = [preds[-1] for preds in val_preds if len(preds) > 0]

val_true_intents_text = [label_list[id] for id in val_true_intents]
val_pred_intents_text = [label_list[id] for id in val_pred_intents]

intent_f1_micro = sklearn_f1(val_true_intents_text, val_pred_intents_text, average='micro')
intent_precision_micro = sklearn_precision(val_true_intents_text, val_pred_intents_text, average='micro')
intent_recall_micro = sklearn_recall(val_true_intents_text, val_pred_intents_text, average='micro')

print("\nValidation Intent Detection Metrics (Micro):")
print(f"Precision: {intent_precision_micro:.4f}")
print(f"Recall: {intent_recall_micro:.4f}")
print(f"F1 Score: {intent_f1_micro:.4f}")

intent_f1_macro = sklearn_f1(val_true_intents_text, val_pred_intents_text, average='macro')
intent_precision_macro = sklearn_precision(val_true_intents_text, val_pred_intents_text, average='macro')
intent_recall_macro = sklearn_recall(val_true_intents_text, val_pred_intents_text, average='macro')

print("\nValidation Intent Detection Metrics (Macro):")
print(f"Precision: {intent_precision_macro:.4f}")
print(f"Recall: {intent_recall_macro:.4f}")
print(f"F1 Score: {intent_f1_macro:.4f}")

print("\nValidation Intent Detection Classification Report (sklearn):")
print(sklearn_classification_report(val_true_intents_text, val_pred_intents_text, digits=4))

torch.cuda.empty_cache()

test_predictions, test_label_ids = predict_in_batches(trainer, tokenized_datasets['test'], batch_size=16)

test_true_labels, test_preds = get_labels(test_predictions, test_label_ids)

test_true_labels_text = id_to_label(test_true_labels, label_list)
test_preds_text = id_to_label(test_preds, label_list)

test_f1_micro = seq_f1(test_true_labels_text, test_preds_text, average='micro')
test_precision_micro = seq_precision(test_true_labels_text, test_preds_text, average='micro')
test_recall_micro = seq_recall(test_true_labels_text, test_preds_text, average='micro')

print("\nTest Slot Filling Metrics (Micro):")
print(f"Precision: {test_precision_micro:.4f}")
print(f"Recall: {test_recall_micro:.4f}")
print(f"F1 Score: {test_f1_micro:.4f}")

test_f1_macro = seq_f1(test_true_labels_text, test_preds_text, average='macro')
test_precision_macro = seq_precision(test_true_labels_text, test_preds_text, average='macro')
test_recall_macro = seq_recall(test_true_labels_text, test_preds_text, average='macro')

print("\nTest Slot Filling Metrics (Macro):")
print(f"Precision: {test_precision_macro:.4f}")
print(f"Recall: {test_recall_macro:.4f}")
print(f"F1 Score: {test_f1_macro:.4f}")


print("\nTest Slot Filling Classification Report:")
print(classification_report(test_true_labels_text, test_preds_text, digits=4))

test_true_intents = [labels[-1] for labels in test_true_labels if len(labels) > 0]
test_pred_intents = [preds[-1] for preds in test_preds if len(preds) > 0]

test_true_intents_text = [label_list[id] for id in test_true_intents]
test_pred_intents_text = [label_list[id] for id in test_pred_intents]


test_intent_f1_micro = sklearn_f1(test_true_intents_text, test_pred_intents_text, average='micro')
test_intent_precision_micro = sklearn_precision(test_true_intents_text, test_pred_intents_text, average='micro')
test_intent_recall_micro = sklearn_recall(test_true_intents_text, test_pred_intents_text, average='micro')

print("\nTest Intent Detection Metrics (Micro):")
print(f"Precision: {test_intent_precision_micro:.4f}")
print(f"Recall: {test_intent_recall_micro:.4f}")
print(f"F1 Score: {test_intent_f1_micro:.4f}")

test_intent_f1_macro = sklearn_f1(test_true_intents_text, test_pred_intents_text, average='macro')
test_intent_precision_macro = sklearn_precision(test_true_intents_text, test_pred_intents_text, average='macro')
test_intent_recall_macro = sklearn_recall(test_true_intents_text, test_pred_intents_text, average='macro')

print("\nTest Intent Detection Metrics (Macro):")
print(f"Precision: {test_intent_precision_macro:.4f}")
print(f"Recall: {test_intent_recall_macro:.4f}")
print(f"F1 Score: {test_intent_f1_macro:.4f}")

print("\nTest Intent Detection Classification Report (sklearn):")
print(sklearn_classification_report(test_true_intents_text, test_pred_intents_text, digits=4))

decoded_text = tokenizer.decode(tokenized_datasets['train'][0]['input_ids'], skip_special_tokens=True)
print("\nمتن بازگردانی‌شده:")
print(decoded_text)
original_text = tokenized_datasets['train'][0]['text']
print("\nمتن اصلی:")
print(original_text)



Validation Slot Filling Metrics (Micro):
Precision: 0.1908
Recall: 0.1784
F1 Score: 0.1844

Validation Slot Filling Metrics (Macro):
Precision: 0.0913
Recall: 0.0885
F1 Score: 0.0779

Validation Slot Filling Classification Report:
                    precision    recall  f1-score   support

        alarm_type     0.0000    0.0000    0.0000         1
       artist_name     0.0000    0.0000    0.0000         2
     business_name     0.0000    0.0000    0.0000         6
     business_type     1.0000    0.2000    0.3333         5
     change_amount     0.0000    0.0000    0.0000         2
        color_type     0.0000    0.0000    0.0000         1
     currency_name     0.0000    0.0000    0.0000         1
              date     0.2292    0.3929    0.2895        28
   definition_word     0.3333    0.5000    0.4000         4
       device_type     0.0000    0.0000    0.0000         2
     email_address     0.0000    0.0000    0.0000         1
      email_folder     0.0000    0.0000    0.00

/usr/local/lib/python3.10/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0


Test Slot Filling Metrics (Micro):
Precision: 0.1884
Recall: 0.1405
F1 Score: 0.1610

Test Slot Filling Metrics (Macro):
Precision: 0.0739
Recall: 0.0891
F1 Score: 0.0676

Test Slot Filling Classification Report:
                      precision    recall  f1-score   support

         artist_name     0.0000    0.0000    0.0000         1
      audiobook_name     0.0000    0.0000    0.0000         1
       business_name     0.0000    0.0000    0.0000         5
       business_type     0.0000    0.0000    0.0000         3
          color_type     0.0000    0.0000    0.0000         4
        cooking_type     0.0000    0.0000    0.0000         2
       currency_name     0.0000    0.0000    0.0000         3
                date     0.2632    0.2500    0.2564        20
     definition_word     0.1667    0.3333    0.2222         3
         device_type     0.2500    0.2500    0.2500         4
          event_name     0.0000    0.0000    0.0000        17
           food_type     0.2500    0.1667

/usr/local/lib/python3.10/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarnin